# NB14 — Final evidence integration and candidate prioritisation
**Tiers (rules fixed in the blueprint; each component is shown separately, no weights):**
1. adjusted (m3) per-gene survival HR concordant in direction in all cohorts **and** meta FDR < 0.05;
2. expressed in malignant cells (M membership; detected in >= 10% of malignant cells in >= 50% of eligible tumours,
   in `min_sc_datasets` dataset(s));
3. stable when any one cohort is left out (needs >= 3 cohorts; otherwise recorded as not evaluable);
4. low heterogeneity (I2 < 50%).
**Tier A** = all four; **Tier B** = two or three; **Tier C** = one or none (a component that is "not evaluable" does not count).
Literature status (NB08) is appended as an annotation and **never** changes tier.

Also builds: the **evidence map** (conclusion -> supporting files and the numbers read from them), the H1/H2/H3 summary, the
alternatives table (why other findings were not the primary story), and a machine-readable `results_summary.json`.
Every number below is read from files written by earlier notebooks.

In [ ]:
import os, sys, json
ROOT = os.environ.get("GBM_ROOT", "/content/drive/MyDrive/GBM_MultiCohort_Research")
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Not in Colab; GBM_ROOT =", ROOT)
sys.path.insert(0, os.path.join(ROOT, "00_Project_Config"))
from gbm_common import *
stop_if_smoke("NB14 integrates full-run results.")
verify_frozen()
FIN = P("final"); T = CFG["tiers"]

In [ ]:
g = pd.read_csv(P("clinical", "gene_cox_meta_M_genes.csv"))
for col in ["direction_concordant", "loo_stable"]:
    g[col] = g[col].map({True: True, False: False, "True": True, "False": False})
c1 = g.direction_concordant.astype(bool) & (g.meta_FDR < T["fdr"])
c2 = pd.Series(True, index=g.index)                    # all genes here are in M by construction
c3 = g.loo_stable              # NaN when < 3 cohorts (not evaluable)
c4 = g.I2 < T["i2_max"]
comp = pd.DataFrame({"C1_adjusted_survival_concordant_FDR": c1, "C2_malignant_expressed": c2,
                     "C3_leave_one_cohort_out_stable": c3, "C4_low_heterogeneity": c4})
n_true = comp.fillna(False).astype(bool).sum(axis=1)
all_eval = comp.notna().all(axis=1)
g = pd.concat([g, comp], axis=1)
g["n_components_met"] = n_true.values
g["tier"] = np.where((n_true == 4) & all_eval, "A", np.where(n_true >= 2, "B", "C"))
g["notes"] = np.where(comp.C3_leave_one_cohort_out_stable.isna(), "component 3 not evaluable (<3 cohorts)", "")
lit_path = P("novelty", "literature_counts.csv")
if lit_path.exists() and not SMOKE:
    lit = pd.read_csv(lit_path).reindex(columns=["gene", "n_any", "n_prognosis_expression", "n_functional", "n_neural_context", "provisional_label"])
    g = g.merge(lit, on="gene", how="left")
    g["literature_note"] = "provisional counts only; manual reading of recent_pmids_for_manual_review.csv required"
g = g.sort_values(["tier", "meta_p"])
g.to_csv(FIN / "candidate_table_tiered.csv", index=False)
print(g.tier.value_counts().to_string())
print(g.head(15)[["gene", "tier", "meta_logHR", "meta_FDR", "I2", "direction_concordant", "loo_stable"]].round(4).to_string(index=False))

In [ ]:
def load_json(p):
    return json.loads(p.read_text()) if p.exists() else None

h1 = load_json(P("clinical", "H1_verdict.json"))
manifest = load_json(P("logs", "run_manifest.json")) or {}
cm = pd.read_csv(P("clinical", "cox_meta_analysis.csv")) if (P("clinical") / "cox_meta_analysis.csv").exists() else pd.DataFrame()
conc = pd.read_csv(P("meta_analysis", "H2_control_type_concordance.csv")) if (P("meta_analysis") / "H2_control_type_concordance.csv").exists() else pd.DataFrame()
mk_enr = pd.read_csv(P("meta_analysis", "H2_marker_enrichment.csv")) if (P("meta_analysis") / "H2_marker_enrichment.csv").exists() else pd.DataFrame()
sens = pd.read_csv(P("sensitivity", "sensitivity_results.csv")) if (P("sensitivity") / "sensitivity_results.csv").exists() else pd.DataFrame()
orth = load_json(P("orthogonal", "orthogonal_status.json"))

# H2 rule from the blueprint: S-gene effect-size Spearman < 0.7 OR marker enrichment above chance (one-sided Fisher, BH < 0.05, OR > 1)
h2 = {"evaluable": bool(len(conc))}
if len(conc):
    sg = conc[conc.gene_set == "S genes"]
    h2["S_gene_min_spearman"] = float(sg.spearman_rho.min()); h2["criterion_concordance_lt_0.7"] = bool((sg.spearman_rho < 0.7).any())
if len(mk_enr):
    pcol = "fisher_FDR" if "fisher_FDR" in mk_enr.columns else "fisher_p"      # BH across cohort x marker set (NB05)
    hit = mk_enr[(mk_enr[pcol] < 0.05) & (mk_enr.odds_ratio > 1)]
    h2["marker_sets_enriched"] = hit[["cohort", "marker_set", "odds_ratio", "fisher_p", pcol]].round(4).to_dict("records")
    h2["criterion_marker_enrichment"] = bool(len(hit))
h2["verdict"] = ("supported" if (h2.get("criterion_concordance_lt_0.7") or h2.get("criterion_marker_enrichment")) else "not_supported") if h2["evaluable"] else "not_evaluable"

n_cohort_info = {k: v for k, v in manifest.get("validation", {}).items()}
summary = {"generated_utc": now_utc(), "H1": h1, "H2": h2, "H3": (orth or {}).get("IvyGAP_H3"),
           "tier_counts": g.tier.value_counts().to_dict(), "n_M_genes_tested_per_gene": int(len(g)),
           "single_cell": manifest.get("single_cell", {}).get("summary"), "validation_cohorts": n_cohort_info,
           "sensitivity": manifest.get("sensitivity", {}).get("summary"), "orthogonal_status": orth,
           "limitations": ["marker-score purity proxies, not DNA-based purity", "one single-cell dataset with an approximate CNV-based malignant call",
                           "CGGA and TCGA differ in ancestry, treatment and platform", "association only; no experimental validation",
                           "meta-analysis with 2-3 cohorts is descriptive"]}
(FIN / "results_summary.json").write_text(json.dumps(summary, indent=2, default=float))
print(json.dumps({k: summary[k] for k in ["H1", "H2", "H3", "tier_counts"]}, indent=1, default=float)[:2500])

In [ ]:
# evidence map: each conclusion -> files and the key numbers read from them
rows = []
def add(conclusion, evidence_files, key_numbers, status):
    rows.append({"conclusion": conclusion, "evidence_files": "; ".join(str(Path(f).relative_to(ROOT)) if str(f).startswith(str(ROOT)) else str(f) for f in evidence_files),
                 "key_numbers": key_numbers, "status": status})
if h1:
    add("H1: adjustment changes the reading of S; M carries the adverse association, R does not",
        [P("clinical", "cox_by_cohort.csv", mkdir=False), P("clinical", "cox_meta_analysis.csv", mkdir=False), P("clinical", "H1_verdict.json", mkdir=False)],
        " | ".join(h1["reasons"]), h1["verdict"])
if h2["evaluable"]:
    add("H2: tumour-vs-normal DEGs depend on control type / non-malignant cell content",
        [P("meta_analysis", "H2_control_type_concordance.csv", mkdir=False), P("meta_analysis", "H2_marker_enrichment.csv", mkdir=False)],
        f"min S-gene Spearman across control pairs = {h2.get('S_gene_min_spearman', float('nan')):.2f}; enriched marker sets: {len(h2.get('marker_sets_enriched', []))}", h2["verdict"])
if len(sens):
    mm = sens[sens.predictor == "M"]
    add("Robustness of the M association", [P("sensitivity", "sensitivity_results.csv", mkdir=False)],
        f"M criterion met in {int(mm.M_criterion_met.sum())} of {len(mm)} variants", "see table")
add("Candidate genes (tiers)", [FIN / "candidate_table_tiered.csv"], str(g.tier.value_counts().to_dict()), "descriptive")
add("H3: M enriched over R in infiltrative regions (exploratory)", [P("orthogonal", "H3_IvyGAP_mixed_model_M_minus_R.csv", mkdir=False)], str((orth or {}).get("IvyGAP_H3")), "exploratory")
emap = pd.DataFrame(rows); emap.to_csv(FIN / "evidence_map.csv", index=False)
print(emap.to_string(index=False))

alt_rows = [
    {"alternative_finding": "Top per-gene survival hits among M genes", "why_not_primary": "per-gene results are many-test, small effects; the pre-specified question is score-level (H1)", "file": "candidate_table_tiered.csv"},
    {"alternative_finding": "Pathway enrichment of meta-DEGs", "why_not_primary": "dominated by established GBM themes and possible composition themes (NB06 tags)", "file": "07_Functional_Enrichment"},
    {"alternative_finding": "WGCNA modules", "why_not_primary": "descriptive; centrality is not causal evidence", "file": "08_Network_Analysis"},
]
pd.DataFrame(alt_rows).to_csv(FIN / "alternatives_not_selected.csv", index=False)
manifest_update("final", "results_summary", str(FIN / "results_summary.json"))
print("\nNB14 complete. Next: NB15.")